## 缓存机制

In [4]:
import time
from operator import add
from typing import TypedDict, Annotated

from langgraph.cache.memory import InMemoryCache
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import CachePolicy
from loguru import logger


# 1、声明状态
class OverAllState(TypedDict):
    user: str
    # 执行次数
    invoke_counts: Annotated[int, add]


# 2、节点
def node_a(state: OverAllState) -> OverAllState:
    logger.info("node_a被调用 user:{}", state["user"])
    # 模拟耗时任务
    time.sleep(3)
    logger.info("node_a 耗时操作执行完毕")
    return {
        # 执行次数 + 1
        "invoke_counts": 1
    }


# 3、建图
builder = StateGraph(state_schema=OverAllState)
# cache_policy 缓存，设置超时时间10s
builder.add_node("node_a", node_a, cache_policy=CachePolicy(ttl=10))

builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

# 4、编译图
graph = builder.compile(cache=InMemoryCache())

logger.info("首次调用图")
logger.info("运行结果 {}\n\n", graph.invoke({"user": "小红", "invoke_counts": 0}))
logger.info("相同输入再次调用图")
logger.info("运行结果 {}\n\n", graph.invoke({"user": "小红", "invoke_counts": 0}))

logger.info("不同输入调用图")
logger.info("运行结果 {}\n\n", graph.invoke({"user": "小明", "invoke_counts": 0}))
logger.info("不同输入再次调用图")
logger.info("运行结果 {}\n\n", graph.invoke({"user": "小明", "invoke_counts": 0}))

time.sleep(11)
logger.info("不同输入调用图")
logger.info("运行结果 {}\n\n", graph.invoke({"user": "小明", "invoke_counts": 0}))
logger.info("不同输入再次调用图")
logger.info("运行结果 {}\n\n", graph.invoke({"user": "小明", "invoke_counts": 0}))

2026-07-22 20:45:26.963 | INFO     | __main__:<module>:42 - 首次调用图
2026-07-22 20:45:26.966 | INFO     | __main__:node_a:21 - node_a被调用 user:小红
2026-07-22 20:45:29.967 | INFO     | __main__:node_a:24 - node_a 耗时操作执行完毕
2026-07-22 20:45:29.973 | INFO     | __main__:<module>:43 - 运行结果 {'user': '小红', 'invoke_counts': 1}


2026-07-22 20:45:29.976 | INFO     | __main__:<module>:44 - 相同输入再次调用图
2026-07-22 20:45:29.986 | INFO     | __main__:<module>:45 - 运行结果 {'user': '小红', 'invoke_counts': 1}


2026-07-22 20:45:29.990 | INFO     | __main__:<module>:47 - 不同输入调用图
2026-07-22 20:45:29.992 | INFO     | __main__:node_a:21 - node_a被调用 user:小明
2026-07-22 20:45:32.993 | INFO     | __main__:node_a:24 - node_a 耗时操作执行完毕
2026-07-22 20:45:32.997 | INFO     | __main__:<module>:48 - 运行结果 {'user': '小明', 'invoke_counts': 1}


2026-07-22 20:45:32.998 | INFO     | __main__:<module>:49 - 不同输入再次调用图
2026-07-22 20:45:33.003 | INFO     | __main__:<module>:50 - 运行结果 {'user': '小明', 'invoke_counts': 1}


2026-07-22 20:45:4

## 解析运行结果
### 1、相同输入，在缓存没过期时 第二次执行，走的时缓存，因为可以看到对应节点第二次调用没有输出
 node_a被调用xxx，最后的一次调用缓存过期后又输出了